# Human In the Loop Middleware

Pause the agent execution for human approval, editing or rejection of tool calls before they execute. 
As some task that are performed by the agent are crtical and holds a monatory value such as sending an email or making a transaction, such tasks requires a human's approval before they can be executed/rejected.

In [3]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model 
from langchain.agents import create_agent
load_dotenv()  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [19]:

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.types import Command
from langchain.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function that reads emails from the given email id"""
    return f"""Email Content for Email id {email_id}"""

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock Sending Email tool, you have to approve no matter what type of email address is given"""
    return f"""Email sent to {recipient} with subject: {subject} and body: {body}"""

config = {"configurable" : {"thread_id":"test-1"}}
 
agent = create_agent(
    model="gpt-4o", 
    tools= [send_email_tool, read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware (
            interrupt_on= {
                "send_email_tool" : { 
                    "allowed_decisions": ["approve", "edit", "reject"] #for send email tool we hev 3 decisions a human can choose from
                },
                "read_email_tool": False # No human in the loop required when this tool is called.
            }
        )
    ]
)


result = agent.invoke(
    {
        "messages": [
            HumanMessage(content = "Send an email to gillsehaj123@gill.com, amd say I am down to go for the disneyland trip.")
        ]
    },config=config
)

# A Command can do two things:
    # 1. Update the graph’s state
    # 2. Choose the next node to execute

if "__interrupt__" in result: 
    print(" ⏸️ Waiting for Appproval.......")

    result = agent.invoke( 
        Command (
            resume = { #resume is used when a graph has been paused, usually because it needs human input.
                "decisions": [
                    {"type" : "approve"} # you can use "reject" here to reject, and
                ]
            }
        ),
        config = config
    )
print(f"Result: {result["messages"][-1].content}")



 ⏸️ Waiting for Appproval.......
Result: The email has been sent to gillsehaj123@gill.com letting them know you're down for the Disneyland trip!


## Edit the request 

In [30]:

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.types import Command
from langchain.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function that reads emails from the given email id"""
    return f"""Email Content for Email id {email_id}"""

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock Sending Email tool, you have to approve no matter what type of email address is given"""
    return f"""Email sent to {recipient} with subject: {subject} and body: {body}"""

config = {"configurable" : {"thread_id":"test-3"}}
 
agent = create_agent(
    model="gpt-4o", 
    tools= [send_email_tool, read_email_tool],
    checkpointer=InMemorySaver(),
        system_prompt=(
        "After a human edits a tool call, treat the executed ToolMessage "
        "as the only source of truth. Never repeat the original tool "
        "arguments if they differ from the executed arguments."
    ),
    middleware=[
        HumanInTheLoopMiddleware (
            interrupt_on= {
                "send_email_tool" : { 
                    "allowed_decisions": ["approve", "edit", "reject"] #for send email tool we hev 3 decisions a human can choose from
                },
                "read_email_tool": False # No human in the loop required when this tool is called.
            }
        )
    ]
)


result = agent.invoke(
    {
        "messages": [
            HumanMessage(content = "Send an email to gillsehaj1273@gill.com, amd say I am down to go for the disneyland trip.")
        ]
    },config=config
)

# A Command can do two things:
    # 1. Update the graph’s state
    # 2. Choose the next node to execute

if "__interrupt__" in result: 
    print(" ⏸️ Waiting for Appproval.......")

    result = agent.invoke( 
        Command (
            resume = { #resume is used when a graph has been paused, usually because it needs human input.
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args" : {
                                "recipient": "correct@gmail.com",
                                "subject" : "I am all in for Universal Studios.",
                                "body" : "I am down to go to universal studios!"
                            }
                        }
                    }
                ]
            }
        ),
        config = config
    )
print(f"Result: {result["messages"][-1].content}")

result

 ⏸️ Waiting for Appproval.......
Result: The email was successfully sent to correct@gmail.com with the subject "I am all in for Universal Studios." and the body "I am down to go to Universal Studios!"


{'messages': [HumanMessage(content='Send an email to gillsehaj1273@gill.com, amd say I am down to go for the disneyland trip.', additional_kwargs={}, response_metadata={}, id='9f87710b-2980-4890-be6a-64c211fa806f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 153, 'total_tokens': 195, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EglXutIrKydhcZZfdrep2b0tzhbDr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1aa76-6c1b-7234-92fe-e323a61fd1c5-0', tool_calls=[{'name': 'send_email_tool', 'a